In [4]:
import pandas as pd
import numpy as np
import os
import json
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

os.makedirs("./output", exist_ok=True)

df = pd.read_csv("../datasets/CICEVSE2024/Power Consumption/EVSE-B-PowerCombined.csv")

os.makedirs("synthetic_extension", exist_ok=True)

filtered_df = df[df["Attack"].isin(["none", "Backdoor", "syn-flood"])].copy()

output_path = "../datasets/CICEVSE2024/synthetic_extension/EVSE-B-PowerCombined_filtered.csv"
filtered_df.to_csv(output_path, index=False)

In [5]:
print(train_df["Attack"].unique())
print(train_df.columns.tolist())

<ArrowStringArray>
['Backdoor', 'none', 'syn-flood']
Length: 3, dtype: str
['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Attack-Group', 'Label', 'interface']


In [6]:
from sklearn.model_selection import train_test_split
import os

os.makedirs("synthetic_extension", exist_ok=True)

train_df, test_df = train_test_split(
    filtered_df,
    test_size=0.3,
    random_state=42,
    shuffle=True,
    #shuffle=False,
    stratify=filtered_df["Attack"]
    #stratify=None
)

train_df.to_csv("../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv", index=False)
test_df.to_csv("../datasets/CICEVSE2024/synthetic_extension/test_dataset.csv", index=False)

In [7]:
train_dist = train_df["Attack"].value_counts().reindex(["none", "Backdoor", "syn-flood"]).fillna(0).astype(int)
test_dist = test_df["Attack"].value_counts().reindex(["none", "Backdoor", "syn-flood"]).fillna(0).astype(int)

train_pct = (train_dist / train_dist.sum() * 100).round(2)
test_pct = (test_dist / test_dist.sum() * 100).round(2)

result = pd.DataFrame({
    "train_count": train_dist,
    "train_pct": train_pct,
    "test_count": test_dist,
    "test_pct": test_pct,
})

print(result)

           train_count  train_pct  test_count  test_pct
Attack                                                 
none             10054      29.30        4309     29.30
Backdoor         14795      43.12        6342     43.13
syn-flood         9462      27.58        4055     27.57
